# Clase 020 — Álgebra lineal

**Parte 0** · `numpy.linalg`.

> 🎯 Operar con vectores y matrices al nivel necesario para ML. Por qué NUNCA usar `inv`.

> ⏱️ ~90 min

## ⚙️ Setup

In [ ]:
import numpy as np
import time
rng = np.random.default_rng(42)

## 1️⃣ El operador `@` (PEP 465)

Desde Python 3.5, `@` es el operador estándar para **multiplicación matricial** (no elementwise, que es `*`).

```python
C = A @ B           # multiplicación matricial
C = A.dot(B)        # equivalente, sintaxis vieja
C = np.matmul(A, B) # equivalente, función explícita
C = A * B           # ¡elementwise! distinto
```

In [ ]:
A = np.array([[1, 2], [3, 4]])
B = np.array([[5, 6], [7, 8]])

print('A @ B (matricial):')
print(A @ B)
print('\nA * B (elementwise):')
print(A * B)

## 2️⃣ Producto punto vector·vector

In [ ]:
a = rng.normal(0, 1, 100)
b = rng.normal(0, 1, 100)

print(f'a @ b        : {a @ b:.4f}')
print(f'np.dot(a, b) : {np.dot(a, b):.4f}')
print(f'sum(a*b)     : {(a*b).sum():.4f}  ← lo mismo')

## 3️⃣ Resolver `Ax = b`: `solve` vs `inv`

**REGLA**: para resolver `Ax = b`, usa `np.linalg.solve(A, b)`, **NUNCA** `np.linalg.inv(A) @ b`.

**Por qué**:
- `inv` calcula la inversa completa (O(n³) caro)
- `solve` usa descomposición LU (O(n³) pero con constante menor)
- `inv` es **numéricamente inestable** (amplifica errores)
- `solve` no construye la inversa, evita ese error

In [ ]:
N = 500
A = rng.normal(0, 1, (N, N))
b = rng.normal(0, 1, N)

t0 = time.perf_counter(); x_inv = np.linalg.inv(A) @ b; t1 = time.perf_counter()
t2 = time.perf_counter(); x_solve = np.linalg.solve(A, b); t3 = time.perf_counter()

print(f'inv(A) @ b : {(t1-t0)*1000:.1f} ms')
print(f'solve(A,b) : {(t3-t2)*1000:.1f} ms')
print(f'speedup    : {(t1-t0)/(t3-t2):.1f}×')

# Precisión: residual ||Ax - b||
print(f'\nresidual inv  : {np.linalg.norm(A @ x_inv - b):.2e}')
print(f'residual solve: {np.linalg.norm(A @ x_solve - b):.2e}')

## 4️⃣ Diagnóstico estructural

```python
np.linalg.norm(v)         # norma L2
np.linalg.det(A)          # determinante (cuidado: 0 ⇒ singular)
np.linalg.matrix_rank(A)  # rango
np.trace(A)               # traza (suma diagonal)
np.linalg.cond(A)         # número de condición (estabilidad)
```

**`cond(A)` grande (>1e10) ⇒ matriz mal condicionada ⇒ `solve` perderá precisión**.

In [ ]:
v = np.array([3, 4])
print(f'norma L2 [3,4]: {np.linalg.norm(v)}  (= 5)')

M = np.array([[2, 1], [1, 3]])
print(f'\nM = {M.tolist()}')
print(f'det      : {np.linalg.det(M):.4f}')
print(f'rank     : {np.linalg.matrix_rank(M)}')
print(f'trace    : {np.trace(M)}')
print(f'cond     : {np.linalg.cond(M):.2f}')

# Matriz singular
S = np.array([[1, 2], [2, 4]])
print(f'\nMatriz singular:')
print(f'det      : {np.linalg.det(S):.4f}')
print(f'rank     : {np.linalg.matrix_rank(S)}  (no es 2)')

## 5️⃣ SVD — la descomposición universal

**Singular Value Decomposition**: cualquier matriz `M (m,n)` se descompone como:

```
M = U · Σ · Vᵀ
```

- `U (m, m)` — vectores singulares izquierdos (ortonormales)
- `Σ (m, n)` — diagonal de **valores singulares** (decrecientes ≥ 0)
- `Vᵀ (n, n)` — vectores singulares derechos (ortonormales)

**Aplicaciones**: PCA, recomendadores (matriz factorization), compresión de imágenes, pseudo-inversa.

In [ ]:
M = rng.normal(0, 1, (6, 4))
U, s, Vt = np.linalg.svd(M, full_matrices=False)

print(f'M.shape  : {M.shape}')
print(f'U.shape  : {U.shape}')
print(f's        : {s.round(3)}  ← valores singulares decrecientes')
print(f'Vt.shape : {Vt.shape}')

# Reconstrucción: M = U @ diag(s) @ Vt
M_reconstruido = U @ np.diag(s) @ Vt
print(f'\nreconstrucción OK: {np.allclose(M, M_reconstruido)}')

## 6️⃣ Eigenvalores y eigenvectores

Para matriz cuadrada `A`, `A v = λ v` donde `λ` es eigenvalor y `v` eigenvector.

**Base conceptual de PCA**: los eigenvectores de la matriz de covarianza son las direcciones de máxima varianza.

In [ ]:
# Matriz de covarianza simulada (simétrica positiva semi-definida)
X = rng.normal(0, 1, (100, 3))
C = np.cov(X.T)   # (3, 3)

# Para matrices simétricas, usa eigh (más rápido, garantiza eigenvalores reales)
eigvals, eigvecs = np.linalg.eigh(C)
print(f'eigvalues (asc): {eigvals.round(4)}')
print(f'\neigvectors (columnas):')
print(eigvecs.round(3))

# La varianza total = suma de eigenvalores
print(f'\ntraza(C)        : {np.trace(C):.4f}')
print(f'sum(eigenvalues): {eigvals.sum():.4f}  ← igual')

## ✅ Checklist

- [ ] Uso `@` para mult matricial, `*` para elementwise
- [ ] NUNCA uso `inv(A) @ b`, siempre `solve(A, b)`
- [ ] Sé qué retorna SVD y verifico la reconstrucción
- [ ] Uso `eigh` para matrices simétricas
- [ ] Conozco `norm`, `det`, `rank`, `cond`

## 📝 Homework

Ver `README.md`. inv vs solve benchmark, regresión lineal cerrada, SVD, eigen de covarianza.

## 📖 Definiciones y características

**Operador `@`**

Multiplicación matricial (PEP 465). `A @ B` ≡ `np.matmul(A, B)` ≡ `A.dot(B)`. NO confundir con `*` (elementwise). Disponible Python 3.5+.

**Producto punto vs producto matricial**

**Punto** (vector·vector → escalar): `a @ b` = `sum(a*b)`. **Matricial** (matriz @ matriz → matriz): regla "fila por columna". Shapes: `(m,n) @ (n,p) → (m,p)`.

**`np.linalg.solve(A, b)`**

Resuelve `Ax = b` usando descomposición LU. **Siempre preferible a `inv(A) @ b`**: más rápido (no construye inversa), más estable numéricamente, menos memoria.

**SVD (Singular Value Decomposition)**

Descomposición universal: `M = U·Σ·Vᵀ`. U y V son ortogonales; Σ diagonal con valores singulares decrecientes ≥ 0. Base de PCA, recomendadores (matrix factorization), compresión, pseudo-inversa.

**Eigen (eigenvalues/eigenvectors)**

Para `A` cuadrada, `A v = λ v`. `eig` general; `eigh` para matrices simétricas (más rápido, garantiza eigenvalores reales). Base de PCA conceptual.

**Número de condición (`cond`)**

Ratio entre el valor singular más grande y el más pequeño. Mide sensibilidad de la solución a perturbaciones. `cond > 1e10` ⇒ matriz mal condicionada, `solve` perderá precisión.

## ⚠️ Errores comunes

| Síntoma / mensaje | Causa y cómo arreglar |
|---|---|
| `A @ B` falla con `ValueError: shapes ... not aligned` | Las dimensiones internas no coinciden: `(m,n) @ (k,p)` requiere `n == k`. **Fix**: revisa shapes, transpone si necesario (`A @ B.T`). |
| Implementé `inv(A) @ b` y los resultados son raros | `inv` es inestable numéricamente para matrices grandes/mal-condicionadas. **Fix**: usa `np.linalg.solve(A, b)` — más rápido y más preciso. |
| `np.linalg.solve` lanza `LinAlgError: Singular matrix` | `det(A) ≈ 0` — el sistema no tiene solución única. **Fix**: si esperabas el caso, usa `np.linalg.lstsq(A, b)` (least squares para sistemas singulares/sobredeterminados). |
| `A * B` da resultado raro y esperaba `A @ B` | Operador `*` es elementwise (Hadamard product). **Fix**: `A @ B` para multiplicación matricial. |
| SVD devuelve `Vt` no `V` | Por convención NumPy devuelve `V^T` (transpuesta), no `V`. Para reconstruir: `M = U @ diag(s) @ Vt`. Si necesitas `V`: `Vt.T`. |

## ❓ Preguntas frecuentes

**❓ ¿`@`, `np.matmul`, o `np.dot`?**

Para matriz×matriz son equivalentes — `@` es el más legible. `np.dot` tiene comportamiento distinto para arrays >2D (no broadcasting); `@` y `matmul` sí. Usa `@` siempre que puedas.

**❓ ¿`eig` o `eigh`?**

**`eigh`** si la matriz es simétrica (covarianza, kernel matrices, métrica de distancias). Más rápido, garantiza eigenvalores reales. **`eig`** para matrices generales (no simétricas) — puede dar valores complejos.

**❓ ¿Por qué `np.linalg.det` para chequear singularidad es mala idea?**

El determinante es 0 o no-0 sin gradiente útil — para matrices grandes, det puede ser tan chico/grande que cause underflow/overflow numérico. Mejor: `np.linalg.cond(A)` — si > 1e10, problemática.

**❓ ¿Cuándo necesito BLAS/LAPACK?**

NumPy ya los usa por debajo (vía OpenBLAS o MKL). Si tu `np.linalg.solve` parece lento, instala MKL (`pip install mkl`) o usa la build de conda con `mkl`.

**❓ ¿GPU para álgebra lineal?**

**CuPy** (drop-in replacement de NumPy con CUDA), **PyTorch tensors** (`.to('cuda')`), o **JAX**. Para matrices >1000×1000 la GPU vale la pena.

## 🔗 Referencias

- [`numpy.linalg`](https://numpy.org/doc/stable/reference/routines.linalg.html)
- [PEP 465 — `@` operator](https://peps.python.org/pep-0465/)

➡️ **Siguiente:** [021 — Aleatoriedad y semillas](../021-numpy-aleatoriedad-y-semillas/README.md)

## ✅ Soluciones de los ejercicios

A continuación, cada ejercicio de la sección `🧪 Ejercicios` del README resuelto y comentado. Todo el código es **ejecutable sin conexión** (datos sintéticos) e incluye `assert`/`print` para que compruebes el resultado. Intenta resolverlos por tu cuenta antes de mirar la solución.

**Ej. 1 — Producto punto** de dos vectores 100-dim.

In [ ]:
import numpy as np
rng = np.random.default_rng(20)
a = rng.normal(size=100); b = rng.normal(size=100)
assert np.isclose(np.dot(a, b), np.sum(a * b))   # dot == suma de productos elementwise
print('a . b =', round(float(np.dot(a, b)), 4))

**Ej. 2 — Multiplicación matricial** (50,30) @ (30,20).

In [ ]:
A = rng.normal(size=(50, 30)); B = rng.normal(size=(30, 20))
C = A @ B
assert C.shape == (50, 20)
assert np.isclose(C[0, 0], np.sum(A[0, :] * B[:, 0]))   # verificacion manual de C[0,0]
print('shape resultante:', C.shape)

**Ej. 3 — Resuelve el sistema** `Ax = b`.

In [ ]:
A = rng.normal(size=(5, 5)); b = rng.normal(size=5)
x = np.linalg.solve(A, b)
assert np.allclose(A @ x, b)                    # A @ x reconstruye b
print('solucion x:', x.round(3))

**Ej. 4 — `inv(A) @ b` vs `solve(A, b)`** (1000x1000).

In [ ]:
import time
A = rng.normal(size=(1000, 1000)); b = rng.normal(size=1000)
t0 = time.perf_counter(); x1 = np.linalg.inv(A) @ b; t_inv = time.perf_counter() - t0
t0 = time.perf_counter(); x2 = np.linalg.solve(A, b); t_solve = time.perf_counter() - t0
assert np.allclose(x1, x2, atol=1e-6)
print(f'inv@b: {t_inv*1000:.1f} ms | solve: {t_solve*1000:.1f} ms | speedup ~{t_inv/max(t_solve,1e-9):.1f}x')
print('solve es preferible: mas rapido y numericamente mas estable que invertir.')

**Ej. 5 — SVD de una matriz rank 1.**

In [ ]:
u = rng.normal(size=(6, 1)); v = rng.normal(size=(4, 1))
M = u @ v.T                                     # rank 1 por construccion
s = np.linalg.svd(M, compute_uv=False)
print('valores singulares:', s.round(4))
assert s[0] > 1e-6 and np.allclose(s[1:], 0, atol=1e-9)
print('Solo el primer valor singular es no-nulo -> rank 1 confirmado.')